In [1]:
from pathlib import Path
import pandas as pd

In [2]:
TABLES = Path("../04_outputs/tables")

bert_df = pd.read_csv(TABLES / "banglabert_binary_summary.csv")
fgm_ben_df = pd.read_csv(TABLES / "banglabert_fgm_ben_sarc_binary_results.csv")
fgm_bs3_df = pd.read_csv(TABLES / "banglabert_fgm_banglasarc3_binary_results.csv")

In [3]:
bert_test = bert_df[bert_df["split"] == "test"].copy()
bert_test = bert_test[bert_test["dataset"].isin(["ben_sarc_binary", "banglasarc3_binary"])].copy()

bert_test = bert_test[[
    "dataset", "accuracy", "macro_f1", "f1_binary"
]].rename(columns={
    "accuracy": "test_accuracy",
    "macro_f1": "test_macro_f1",
    "f1_binary": "test_f1_binary"
})
bert_test["model"] = "banglabert"

In [4]:
fgm_df = pd.concat([fgm_ben_df, fgm_bs3_df], ignore_index=True)
fgm_test = fgm_df[fgm_df["split"] == "test"].copy()

fgm_test = fgm_test[[
    "dataset", "accuracy", "macro_f1", "f1_binary"
]].rename(columns={
    "accuracy": "test_accuracy",
    "macro_f1": "test_macro_f1",
    "f1_binary": "test_f1_binary"
})
fgm_test["model"] = "banglabert_fgm"

In [5]:
comparison_df = pd.concat([bert_test, fgm_test], ignore_index=True)
comparison_df = comparison_df[[
    "dataset", "model", "test_accuracy", "test_macro_f1", "test_f1_binary"
]].sort_values(["dataset", "model"]).reset_index(drop=True)

comparison_df

,dataset,model,test_accuracy,test_macro_f1,test_f1_binary
0,banglasarc3_binary,banglabert,0.735661,0.735290,0.745192
1,banglasarc3_binary,banglabert_fgm,0.745636,0.745444,0.752427
2,ben_sarc_binary,banglabert,0.796412,0.795731,0.783940
3,ben_sarc_binary,banglabert_fgm,0.809672,0.809611,0.806195


In [6]:
pivot_df = comparison_df.pivot(
    index="dataset",
    columns="model",
    values="test_macro_f1"
).reset_index()

pivot_df["fgm_gain"] = pivot_df["banglabert_fgm"] - pivot_df["banglabert"]
pivot_df

model,dataset,banglabert,banglabert_fgm,fgm_gain
0,banglasarc3_binary,0.735290,0.745444,0.010154
1,ben_sarc_binary,0.795731,0.809611,0.013880


In [7]:
comparison_df.to_csv(TABLES / "fgm_comparison_long.csv", index=False)
pivot_df.to_csv(TABLES / "fgm_comparison_macro_f1.csv", index=False)

print(TABLES / "fgm_comparison_long.csv")
print(TABLES / "fgm_comparison_macro_f1.csv")

../04_outputs/tables/fgm_comparison_long.csv
../04_outputs/tables/fgm_comparison_macro_f1.csv
